# La Firma Sin Sentido: Falsificando ECDSA Sin Clave Privada

**Basado en:** [Faketoshi's Nonsense Signature](https://jimmysong.medium.com/faketoshis-nonsense-signature-8700a44536b5) de Jimmy Song (2018)  
**Crédito matemático:** Andy Poelstra, Greg Maxwell, Pieter Wuille

[← Volver al Esquema Didáctico de ECC](./00-ecc-teachable-scheme.ipynb) | [El Truco de Wright (2016) →](./01-ecc-wright-trick.ipynb)

---

## El Contexto

En noviembre de 2018, una cuenta de Twitter publicó una "firma" de la
clave pública del bloque génesis de Satoshi. Verificaba. La gente entró en pánico.

Pero la firma no valía nada — un truco matemático de salón que
**cualquiera** puede hacer para **cualquier** clave pública, sin conocer la clave privada.

Este es un ataque diferente al [truco del blog de Wright en 2016](./01-ecc-wright-trick.ipynb)
(que reutilizó una firma real de la blockchain). Este es álgebra pura —
y enseña una lección más profunda sobre lo que ECDSA realmente garantiza.

### La analogía de Jimmy Song

> Es como "demostrar" que corriste un maratón en menos de 2 horas empezando
> cerca de la meta.

---

## Parte 1: Cómo Funciona Realmente la Verificación ECDSA

Antes de entender el truco, necesitamos ver la verificación como una ecuación
con variables — no solo como una caja negra que dice "válido" o "inválido".

### La fórmula de verificación

Dada una clave pública $P$, un hash del mensaje $z$ y una firma $(r, s)$:

$$u = z \cdot s^{-1} \bmod N$$
$$v = r \cdot s^{-1} \bmod N$$
$$R' = u \times G + v \times P$$
$$\text{Válido si } R'_x \equiv r \pmod{N}$$

### Contar los grados de libertad

```
Firma normal:                        El truco:
─────────────                        ──────────
DADO: z (hash del mensaje real)      DADO: P (clave pública — de la blockchain)
DADO: P (clave pública)              ELEGIR: u, v (aleatorios)
SECRETO: d (clave privada)           DERIVAR: r, s, z (para satisfacer la fórmula)
SALIDA: (r, s)

El firmante tiene 1 variable         El atacante tiene 2 variables
libre (k) y DEBE satisfacer z.       libres (u, v) y puede ELEGIR z.

Más libertad → puede satisfacer      Pero z es basura, no es un hash
la ecuación sin conocer d.           de ningún mensaje real.
```

### La observación de Pieter Wuille

> "Las firmas ECDSA donde el mensaje no es un hash y es elegido por el
> 'firmante' son inseguras." — Pieter Wuille

Toda la seguridad de ECDSA depende de que el firmante NO controle $z$.
En el momento en que lo hace, la clave privada se vuelve irrelevante.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets

from ecc import *

def ecdsa_verify_z(z: int, r: int, s: int, pub: Point) -> bool:
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u = (z * s_inv) % SECP_N
    v = (r * s_inv) % SECP_N
    R = point_add(scalar_mult(u, G), scalar_mult(v, pub))
    return R.x % SECP_N == r

print("Primitivas criptográficas cargadas.")

## Parte 2: La Falsificación — Paso a Paso

Vamos a falsificar una firma ECDSA "válida" para la clave pública del bloque
génesis de Satoshi. No conocemos la clave privada. Nunca la conoceremos.
Pero la firma verificará.

### El algoritmo

```
1. Elegir u, v aleatorios
2. Calcular R = u×G + v×P         (P es la clave pública objetivo)
3. Fijar r = R.x mod N
4. Fijar s = r / v mod N            (de la fórmula de verificación: v = r/s)
5. Fijar z = u × s mod N            (de la fórmula de verificación: u = z/s)
6. La "firma" es (r, s) para el "hash del mensaje" z
```

### ¿Por qué pasa la verificación?

El verificador calcula:
- $u' = z \cdot s^{-1} = (u \cdot s) \cdot s^{-1} = u$ 
- $v' = r \cdot s^{-1} = r \cdot (v / r) = v$
- $R' = u' \times G + v' \times P = u \times G + v \times P = R$
- $R'_x = R_x = r$ ✓

Diseñamos todo para que la ecuación de verificación se satisfaga trivialmente.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LA FALSIFICACIÓN
# ═══════════════════════════════════════════════════════════════

# Objetivo: clave pública del bloque génesis de Satoshi (cualquiera puede leerla)
GENESIS_PUBKEY_HEX = (
    "04678afdb0fe5548271967f1a67130b7105cd6a828e03909"
    "a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112"
    "de5c384df7ba0b8d578a4c702b6bf11d5f"
)

# Parsear la clave pública sin comprimir (04 || x || y)
pubkey_bytes = bytes.fromhex(GENESIS_PUBKEY_HEX)
px = int.from_bytes(pubkey_bytes[1:33], 'big')
py = int.from_bytes(pubkey_bytes[33:65], 'big')
P = Point(px, py)

# Verificar que P está en la curva
assert (py * py) % SECP_P == (px * px * px + 7) % SECP_P
print("Objetivo: clave pública del bloque génesis de Satoshi")
print(f"  P.x = {hex(px)[:24]}...")
print(f"  P.y = {hex(py)[:24]}...")
print(f"  En la curva: True")
print(f"\nNO conocemos la clave privada.")
print(f"Estamos a punto de falsificar una firma 'válida' de todos modos.")

In [ ]:
# Paso 1: Elegir u y v aleatorios
u = secrets.randbelow(SECP_N - 1) + 1
v = secrets.randbelow(SECP_N - 1) + 1

print("PASO 1: Elegir u, v aleatorios")
print(f"  u = {hex(u)[:20]}...  (aleatorio)")
print(f"  v = {hex(v)[:20]}...  (aleatorio)")

# Paso 2: Calcular R = u×G + v×P
R = point_add(scalar_mult(u, G), scalar_mult(v, P))
print(f"\nPASO 2: R = u×G + v×P")
print(f"  R.x = {hex(R.x)[:24]}...")

# Paso 3: r = R.x mod N
r = R.x % SECP_N
print(f"\nPASO 3: r = R.x mod N")
print(f"  r = {hex(r)[:24]}...")

# Paso 4: s = r/v mod N  (de v = r/s → s = r/v)
v_inv = pow(v, SECP_N - 2, SECP_N)
s = (r * v_inv) % SECP_N
print(f"\nPASO 4: s = r × v⁻¹ mod N")
print(f"  s = {hex(s)[:24]}...")

# Paso 5: z = u×s mod N  (de u = z/s → z = u×s)
z = (u * s) % SECP_N
print(f"\nPASO 5: z = u × s mod N")
print(f"  z = {hex(z)[:24]}...")
print(f"  (Este z es BASURA — no es el hash de ningún mensaje real)")

In [ ]:
# Paso 6: Verificar — ¿pasa?
print("PASO 6: Verificación")
print("=" * 50)

valid = ecdsa_verify_z(z, r, s, P)

print(f"  ecdsa_verify_z(z, r, s, P) = {valid}")
print(f"")
print(f"  La firma VERIFICA. ✓")
print(f"  Para la clave pública del bloque génesis de Satoshi.")
print(f"  Sin conocer la clave privada.")
print(f"")
print(f"  Pero z = {hex(z)[:24]}...")
print(f"  ¿Qué mensaje tiene este hash SHA256? Ninguno.")
print(f"  No es un hash de nada. Es basura algebraica.")
print(f"")
print(f"  Es como mostrar un pase de abordar válido")
print(f"  para un vuelo que no existe.")

---

## Parte 3: Por Qué Esto No Es una Firma Real

### El problema de z

En ECDSA real, $z$ está **determinado por el mensaje**:

$$z = \text{SHA256}(\text{mensaje})$$

El firmante no puede elegir $z$ — está dictado por lo que sea que esté firmando.
El verificador calcula $z$ independientemente a partir del mismo mensaje y comprueba.

En la falsificación, $z$ se derivó al revés a partir de $u$ y $s$. No hay
mensaje. Si preguntas "¿qué mensaje se firmó?", la respuesta es: ninguno.
Tendrías que encontrar un mensaje cuyo hash SHA256 sea igual a ese $z$ — lo cual es
un **ataque de preimagen a SHA256**, que se cree imposible.

### Dónde realmente reside la seguridad

```
Cadena de seguridad de ECDSA:

  Mensaje  ──SHA256──→  z  ──ECDSA──→  (r, s)
     ↑                  ↑                 ↑
  Elegido por        Determinado por   Requiere clave
  el protocolo       el hash            privada para
  (no el firmante)   (no el firmante)   producir dado un z

El truco sin sentido rompe la cadena en z:

  ???  ──???──→  z  ←──DERIVADO DE──  (r, s)
                 ↑
              Elegido por el ATACANTE
              (la seguridad colapsa)
```

### Las señales de alerta en el tweet original

1. Software de verificación personalizado (no una billetera Bitcoin estándar)
2. El "mensaje" era un hash hexadecimal crudo, no un texto legible
3. Sin desafío-respuesta — el "firmante" eligió todo

In [ ]:
# Hagámoslo aún más obvio: falsificar 5 firmas seguidas
# para la misma clave pública, cada una "válida", cada una sin sentido

print("=== Falsificando 5 firmas 'válidas' para la clave de Satoshi ===")
print(f"(Todas sin conocer la clave privada)\n")

for i in range(5):
    u_i = secrets.randbelow(SECP_N - 1) + 1
    v_i = secrets.randbelow(SECP_N - 1) + 1
    R_i = point_add(scalar_mult(u_i, G), scalar_mult(v_i, P))
    r_i = R_i.x % SECP_N
    s_i = (r_i * pow(v_i, SECP_N - 2, SECP_N)) % SECP_N
    z_i = (u_i * s_i) % SECP_N
    valid_i = ecdsa_verify_z(z_i, r_i, s_i, P)
    print(f"  Firma {i+1}: r={hex(r_i)[:14]}... s={hex(s_i)[:14]}... z={hex(z_i)[:14]}... válida={valid_i}")

print(f"\nLas 5 verifican. Las 5 no valen nada.")
print(f"Ninguna corresponde a un mensaje real.")

In [ ]:
# Ahora contrastemos con una firma REAL: el firmante no puede elegir z

print("=== Cómo se ve una prueba REAL ===")
print()

# Mensaje de desafío de Jimmy Song del artículo
challenge = "The Times 19/11/2018 The ups and downs of Downing Street"
z_real = int.from_bytes(hashlib.sha256(challenge.encode()).digest(), 'big')

print(f"Desafío: '{challenge}'")
print(f"z = SHA256(desafío) = {hex(z_real)[:24]}...")
print()
print(f"Para producir un (r, s) válido para ESTE z específico,")
print(f"DEBES conocer la clave privada. Ningún truco lo evita.")
print()
print(f"Las únicas opciones del atacante:")
print(f"  1. Conocer la clave privada d           → puede firmar cualquier cosa")
print(f"  2. Encontrar k tal que (kG).x = r para  → problema del logaritmo discreto")
print(f"     un r específico que satisfaga la      → computacionalmente inviable")
print(f"     ecuación con este z")
print(f"  3. Encontrar otro mensaje con el        → ataque de preimagen a SHA256")
print(f"     mismo z                              → computacionalmente inviable")
print()
print(f"Cuando z lo fija el verificador, no hay atajo.")

---

## Parte 4: Comparación — Dos Formas de Falsificar a Satoshi

| | **2016: Blog Post** (Wright) | **2018: Tweet** (Faketoshi) |
|---|---|---|
| **Técnica** | Reutilizar una firma real de la blockchain | Falsificar una firma sin sentido algebraicamente |
| **Requiere** | Una tx real de Satoshi de la blockchain | Solo la clave pública (también pública) |
| **Matemática** | `SHA256("archivo Sartre") = SHA256(SHA256(modtx))` | Elegir `u,v` → derivar `r,s,z` al revés |
| **El z** | Real (de una transacción de 2009) | Basura (no es un hash de ningún mensaje) |
| **Firma** | Real (copiada de la blockchain) | Falsa (diseñada para pasar la verificación) |
| **Sofisticación** | Moderada (requiere entender internals de tx Bitcoin) | Baja (álgebra de pregrado) |
| **Lección ECDSA** | La verificación no prueba *quién* firmó | La verificación no funciona si el firmante controla *z* |
| **Defensa** | El verificador elige un mensaje de desafío nuevo | *z* debe ser un hash de un mensaje real elegido por el verificador |

### Ambos ataques fallan contra la misma defensa

```
El verificador dice: "Firma este mensaje exacto: [desafío impredecible]"

  → Atacante de replay: no puede, las firmas viejas tienen z incorrecto
  → Atacante sin sentido: no puede, z lo determina el desafío, no él
  → Poseedor real de la clave: produce trivialmente (r, s) válido para cualquier z
```

---

## Puntos Clave

1. **ECDSA no está roto.** Ambos trucos explotan el *protocolo alrededor de* ECDSA, no la matemática en sí.

2. **El hash del mensaje z es sagrado.** Si el firmante elige z, la seguridad desaparece. El verificador debe determinar z (eligiendo el mensaje o acordándolo previamente).

3. **"Firma válida" es necesario pero no suficiente.** La verificación responde: "¿satisface este (r,s) la ecuación para este z y P?" NO responde: "¿produjo el poseedor de la clave privada esto para un mensaje significativo?"

4. **Siempre usa herramientas estándar.** Software de verificación personalizado es una señal de alerta. Si alguien no puede usar una billetera Bitcoin estándar para verificar, pregunta por qué.

---

*Fuentes:*
- *[Faketoshi's Nonsense Signature](https://jimmysong.medium.com/faketoshis-nonsense-signature-8700a44536b5) — Jimmy Song*
- *[Pieter Wuille sobre hashes de mensajes elegidos por el firmante](https://twitter.com/pwuille/status/1063582706288586752)*
- *[Código de falsificación](https://0bin.net/paste/7-tnWL-IgsKqGFcS) — Andy Poelstra, Greg Maxwell, Pieter Wuille*
- *Jimmy Song, [Programming Bitcoin](https://programmingbitcoin.com/) — Capítulos 1-4*